# Retrieval-Augmented Generation over Scientific Literature

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

## The problem: LLMs don't know your papers

Foundation models are trained on a snapshot of the public internet, then
frozen. That means they:

- Have never seen your group's preprints, internal reports, or last week's
  beamtime notes.
- Cannot reliably recall a *specific* number from a *specific* paper, even
  if that paper was in their training set — the model "remembers" facts
  statistically, not verbatim.
- Have no way to update when a new paper is published — short of expensive
  retraining.
- Will happily *make something up* in a confident tone if you ask about
  details they don't actually have. This is the famous "hallucination"
  failure mode, and it is most acute on exactly the things we care about
  in science: numbers, citations, and conditions.

You cannot fix this by asking the model to be more careful. The information
simply isn't in the weights.

## The fix: give the model the documents at query time

**Retrieval-Augmented Generation (RAG)** sidesteps the problem by treating
the LLM as a *reader*, not a *recall engine*. Instead of asking *"what
do you know about X?"*, you do this:

1. **Find the few documents (or paragraphs) most relevant to the question.**
2. **Paste those into the prompt.**
3. **Ask the model to answer using only that text.**

That's the whole idea. The model doesn't need to memorise your corpus —
it just needs to read it.

The hard part is step 1. With a corpus of 8 papers you could glance through
them yourself; with 8 million papers you need a way to find the relevant
handful in milliseconds. That is what the rest of the RAG machinery —
embeddings, vector databases, similarity search — is for.

## Why embeddings work (the 60-second version)

An **embedding** is a fixed-length numeric vector that captures the
*meaning* of a piece of text. The trick: embedding models are trained so
that texts with similar meaning get vectors that are close together in
high-dimensional space, and texts with unrelated meaning get vectors that
are far apart.

Concretely, "lithium-ion cathode capacity" and "Li-ion battery energy
density" share almost no keywords, but their embedding vectors point in
nearly the same direction. Meanwhile "perovskite solar cell" lives in a
different region of space entirely. So the question *"which paragraphs in
my corpus are about Li-ion cathode performance?"* becomes a *geometry*
problem: find the corpus vectors closest to the query vector. That's a
single matrix multiplication for small corpora, or a fast nearest-neighbour
index (FAISS, HNSW, etc.) for large ones.

This is why dense retrieval beats keyword search: it matches on *meaning*,
not on word overlap.

## The full picture

```
                                 ┌────────────────┐
   YOUR DOCUMENTS  ──── chunk ──→│   embedder     │── vectors ──→  vector store
   (PDFs, .md,                   └────────────────┘                (numpy / FAISS / Qdrant)
   notebooks, ...)                                                        │
                                                                          │
                                                                          ▼
   USER QUESTION  ─────────────→ │   embedder     │── query vec ─→  similarity search
                                 └────────────────┘                       │
                                                                          ▼
                                                                  top-k relevant chunks
                                                                          │
                                                                          ▼
                                                              ┌──────────────────────┐
                                                              │ "Answer using ONLY   │
                                                              │  these passages..."  │── LLM ──→  cited answer
                                                              └──────────────────────┘
```

Two phases:
- **Index time** (slow, done once when the corpus changes): chunk every
  document, run the embedder over each chunk, store the vectors.
- **Query time** (fast, every user request): embed the question, retrieve
  the top-k most similar chunks, build a prompt that includes them, send
  to the LLM.

## When to reach for RAG

| You should use RAG when... | You don't need RAG when... |
|---|---|
| Answers must come from *your* documents (papers, internal reports, lab notes) | The question is generic and the LLM's pre-training already covers it well |
| You need *citations* back to source files | Hallucination is acceptable (e.g. brainstorming, code completion) |
| The corpus changes frequently and you can't keep retraining | The corpus fits entirely in the model's context window — just paste it all in |
| Provenance matters (regulators, reproducibility) | You're doing arithmetic / structured computation — give the model a tool, not a document |

**RAG is not** fine-tuning, retraining, or "teaching the model" anything.
The model weights never change. RAG is just a prompt-construction strategy
that happens to involve vector search.

## What you'll build in this notebook

A complete RAG system in ~200 lines, no frameworks:
- A small corpus of (fictional) materials-science abstracts
- Embeddings via `sentence-transformers` (runs on CPU in seconds)
- Cosine-similarity retrieval with `numpy`
- Generation via the Anthropic API with a "cite your sources" system prompt
- A side-by-side comparison: the same question with and without retrieval

By the end you'll have seen every moving part. After that, switching to
LangChain, LlamaIndex, or a hosted vector DB is just an exercise in
swapping components — the *shape* of the pipeline is the same.

---
## Setup

In [ ]:
# !pip install anthropic sentence-transformers numpy

In [ ]:
import os, getpass, numpy as np, json, textwrap

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

import anthropic
from sentence_transformers import SentenceTransformer

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

# A small, fast, open embedding model. Runs on CPU in a few seconds.
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding dim:", embedder.get_sentence_embedding_dimension())

---
## Step 1: The corpus

For the tutorial we use 8 short, fictional abstracts spanning perovskites,
2D materials, and battery cathodes. In a real workflow this is where you'd
load a folder of PDFs (via `pypdf`, `pymupdf`, `unstructured`, …).

In [ ]:
CORPUS = [
    {
        "id": "abs01",
        "title": "Sol-gel synthesis of tetragonal BaTiO3 nanoparticles",
        "text": (
            "We report the synthesis of phase-pure tetragonal BaTiO3 nanoparticles by "
            "a sol-gel route. Calcination at 750 C for 6 h in oxygen yielded an average "
            "crystallite size of 38 nm with a Curie temperature of 128 C measured by DSC."
        ),
    },
    {
        "id": "abs02",
        "title": "Hydrothermal growth of SrTiO3 cubes",
        "text": (
            "Cubic SrTiO3 microparticles (1-3 microns) were grown hydrothermally at 200 C "
            "for 24 h from SrCl2 and TiO2 precursors in 8 M KOH. The bandgap was measured "
            "as 3.25 eV from diffuse reflectance."
        ),
    },
    {
        "id": "abs03",
        "title": "Mechanical exfoliation of 2H-MoS2 monolayers",
        "text": (
            "Monolayer 2H-MoS2 was prepared by mechanical exfoliation from natural single "
            "crystals onto SiO2/Si substrates. Photoluminescence at 1.88 eV confirmed "
            "the indirect-to-direct bandgap transition characteristic of single-layer MoS2."
        ),
    },
    {
        "id": "abs04",
        "title": "CVD growth of large-area WSe2",
        "text": (
            "Large-area WSe2 monolayers were grown by chemical vapor deposition on sapphire "
            "at 850 C using WO3 and selenium powders. Triangular domains up to 200 microns "
            "were observed; field-effect mobility reached 45 cm^2/V/s at room temperature."
        ),
    },
    {
        "id": "abs05",
        "title": "Layered NMC811 cathode for Li-ion batteries",
        "text": (
            "LiNi0.8Mn0.1Co0.1O2 (NMC811) was synthesized by co-precipitation followed by "
            "lithiation at 780 C in oxygen. The cathode delivered 198 mAh/g at 0.1C between "
            "2.7-4.3 V vs Li/Li+, with 87% capacity retention after 100 cycles."
        ),
    },
    {
        "id": "abs06",
        "title": "Olivine LiFePO4 via solid-state synthesis",
        "text": (
            "Carbon-coated LiFePO4 was prepared by solid-state reaction of Li2CO3, FePO4, "
            "and sucrose at 700 C for 10 h under argon. The material delivered 158 mAh/g "
            "at C/10 with excellent cycling stability over 500 cycles."
        ),
    },
    {
        "id": "abs07",
        "title": "Pulsed laser deposition of BiFeO3 thin films",
        "text": (
            "BiFeO3 thin films were grown on (001) SrTiO3 by pulsed laser deposition at "
            "650 C in 100 mTorr O2. Films were 80 nm thick; XRD showed a single rhombohedral "
            "phase with a remnant polarization of 65 microC/cm^2."
        ),
    },
    {
        "id": "abs08",
        "title": "Spinel Co3O4 nanowires by hydrothermal growth",
        "text": (
            "Spinel Co3O4 nanowires (diameter 80 nm, length 5 microns) were grown "
            "hydrothermally at 180 C from Co(NO3)2 and urea, then annealed at 350 C in air. "
            "The Neel temperature was determined to be 33 K from SQUID measurements."
        ),
    },
]
print(f"Corpus has {len(CORPUS)} documents")

---
## Step 2: Chunk the corpus

For longer documents you'd split each into ~500-token chunks. Here every
abstract already fits in one chunk — we'll keep one row per abstract for clarity,
but the *index structure* is the part that generalizes.

In [ ]:
# In real corpora: split each doc into overlapping windows (e.g. 512 tokens with 64 overlap).
# Here each abstract is small enough to be its own chunk.
chunks = [{
    "doc_id": d["id"],
    "title":  d["title"],
    "text":   d["text"],
} for d in CORPUS]

print(f"{len(chunks)} chunks total")

---
## Step 3: Embed everything

Each chunk becomes a 384-dimensional vector that captures its *meaning*.
Two chunks with similar meaning will have nearby vectors even if they share no
keywords (the magic of dense retrieval).

In [ ]:
# Encode all chunks at once. .encode() returns a (n_chunks, dim) numpy array.
texts = [f"{c['title']}. {c['text']}" for c in chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True)
print("Embeddings shape:", embeddings.shape)

**Why normalize?** Cosine similarity = dot product when both vectors are
unit-length. Normalizing lets us replace "compute cosine for all pairs" with a
single matrix multiplication.

---
## Step 4: Retrieval

Given a query, embed it the same way and find the nearest chunks by cosine
similarity. For 8 chunks we just use numpy; for 8 million you'd reach for FAISS,
HNSW, Qdrant, or Pinecone — but the API is the same.

In [ ]:
def retrieve(query: str, k: int = 3):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q                             # cosine sims
    top = np.argsort(-scores)[:k]
    return [(chunks[i], float(scores[i])) for i in top]

for chunk, score in retrieve("How do I make a perovskite oxide for ferroelectric devices?"):
    print(f"[{score:.3f}] {chunk['doc_id']}: {chunk['title']}")

Notice how chunks about BaTiO3 and BiFeO3 (both ferroelectric perovskites)
score highest — even though the query never mentions either compound. That's
the point of dense retrieval: meaning, not keywords.

---
## Step 5: The hallucination baseline

Before adding retrieval, let's see what the model does *without* the corpus.
If we ask about a specific number from a specific (fictional) paper, the model
has no choice but to guess — or to refuse.

In [ ]:
QUESTION = ("In the BaTiO3 nanoparticle paper using a sol-gel route with calcination "
            "at 750 C, what was the reported average crystallite size?")

resp = client.messages.create(
    model=MODEL, max_tokens=400,
    messages=[{"role": "user", "content": QUESTION}],
)
print("[NO RETRIEVAL]\n")
print(resp.content[0].text)

You'll typically see one of:
- A confident wrong number ("approximately 50 nm")
- A plausible-sounding range ("typically 20-100 nm depending on conditions")
- A generic refusal

None of these are useful in a scientific pipeline.

---
## Step 6: RAG generation

Now we *retrieve first*, then build a prompt that gives the model the relevant
chunks and instructs it to answer **only from the provided context**.

In [ ]:
def rag_answer(question: str, k: int = 3) -> str:
    hits = retrieve(question, k=k)
    context = "\n\n".join(
        f"[{c['doc_id']}] {c['title']}\n{c['text']}"
        for c, _ in hits
    )
    system = (
        "You answer materials-science questions using ONLY the provided source passages. "
        "If the answer is not contained in the sources, say so explicitly. "
        "Cite each fact with its [doc_id] tag."
    )
    user = f"Sources:\n\n{context}\n\nQuestion: {question}"
    resp = client.messages.create(
        model=MODEL, max_tokens=400, system=system,
        messages=[{"role": "user", "content": user}],
    )
    return resp.content[0].text

print("[WITH RAG]\n")
print(rag_answer(QUESTION))

Now you'll get the *actual* number from the abstract (38 nm) with a
citation back to `abs01`. The model isn't being clever — it's *reading*. That's
the whole job.

---
## Step 7: When retrieval fails (and why honesty matters)

What happens when the corpus simply doesn't contain the answer?

In [ ]:
OFF_TOPIC = "What is the magnetic ordering temperature of YBa2Cu3O7?"

print("[WITH RAG, off-topic question]\n")
print(rag_answer(OFF_TOPIC))

A well-instructed RAG system says **"the sources don't contain this"** rather than
making something up. This is the single most important behavior to test for —
silent fabrication is the failure mode that breaks scientific trust.

---
## Step 8: Inspecting retrieval quality

Bad RAG output is almost always *retrieval* failure, not *generation* failure.
Always log what you retrieved alongside the answer:

In [ ]:
def rag_with_trace(question: str, k: int = 3):
    hits = retrieve(question, k=k)
    answer = rag_answer(question, k=k)
    print("Question:", question)
    print("\nRetrieved:")
    for c, s in hits:
        print(f"  [{s:.3f}] {c['doc_id']}: {c['title']}")
    print("\nAnswer:", answer)
    print("-" * 70)

for q in [
    "Which papers report room-temperature electrical mobility?",
    "Compare the synthesis temperatures used for the Li-ion cathodes.",
    "What ferroelectric materials are described and what are their key properties?",
]:
    rag_with_trace(q)

---
## Where to go next

This implementation is intentionally minimalist. In production you'd add:

| Concern | What to add |
|---|---|
| Long documents | Chunking with overlap (`langchain_text_splitters`, `unstructured`) |
| Large corpora | A real vector DB (FAISS, Qdrant, Weaviate, pgvector) |
| Better retrieval | Hybrid (BM25 + dense), reranking with a cross-encoder |
| Updated knowledge | Incremental indexing as new papers arrive |
| Evaluation | A held-out QA set; measure "did the answer cite the right doc?" |

But the core loop — *embed, retrieve, generate, cite* — stays the same, and
that loop is what the next notebooks build agents on top of.